# Hobby Matchmaker - a content-based recommender (117 hobbies)

**Day 2 NLP project (YDL 2026), variant 2.** A different angle from the Reddit/Word2Vec
notebook: here the hobbies and their **descriptions** come straight from a ready dataset
([`alperugurcan/Hobbies`](https://huggingface.co/datasets/alperugurcan/Hobbies) on Hugging Face),
so the vocabulary is fully **data-driven** - no hand-written list.

Each hobby is described by `tags` (e.g. *creative, visual, artistic*) plus attributes
(difficulty, cost, time, physical level, sociability, location, age group, equipment). I turn
those into a **vector per hobby** (tf-idf on the tags + one-hot on the attributes), and similar
vectors = hobbies that are alike **in character**. Cosine similarity then powers a recommender
and an interactive game.

> Note: this is **content-based** (similar *by nature*), unlike the Reddit notebook which is
> **behavioural** (similar = *the same people do them*). Two valid, complementary notions of
> "similar".

In [1]:
# HTML/CSS/JS template for the Hobby Matchmaker game (used in section 5)
GAME_TEMPLATE = r'''<!doctype html>
<html lang="en"><head>
<meta charset="utf-8"><meta name="viewport" content="width=device-width, initial-scale=1">
<title>Hobby Matchmaker</title>
<style>
  * { box-sizing:border-box; }
  body { margin:0; min-height:100vh; font-family:-apple-system,Segoe UI,Roboto,Helvetica,Arial,sans-serif;
         color:#1f2330; background:linear-gradient(135deg,#6a8dff 0%,#9b5de5 50%,#ff7eb3 100%);
         background-attachment:fixed; padding:32px 16px; }
  .wrap { max-width:980px; margin:0 auto; }
  h1 { text-align:center; color:#fff; margin:4px 0 2px; font-size:34px; text-shadow:0 2px 14px #0003; }
  .sub { text-align:center; color:#ffffffdd; margin:0 0 22px; font-size:15px; }
  .panel { background:#ffffffee; border-radius:20px; padding:22px 22px 26px;
           box-shadow:0 18px 50px #0000003a; backdrop-filter:blur(6px); }
  .grp { margin:14px 0 6px; font-weight:700; font-size:14px; display:flex; align-items:center; gap:8px; }
  .dot { width:11px; height:11px; border-radius:50%; display:inline-block; }
  .chips { display:flex; flex-wrap:wrap; gap:9px; }
  .chip { padding:7px 12px; border-radius:999px; border:2px solid #e6e6ee; background:#fff;
          cursor:pointer; font-size:13px; user-select:none; transition:.16s; }
  .chip:hover { transform:translateY(-2px); box-shadow:0 6px 16px #0002; }
  .chip.on { color:#fff; border-color:transparent; box-shadow:0 6px 16px #0003; }
  .bar { display:flex; gap:12px; justify-content:center; margin-top:22px; flex-wrap:wrap; }
  button { border:0; border-radius:999px; padding:13px 26px; font-size:15px; font-weight:700; cursor:pointer; transition:.16s; }
  .go { background:#1f2330; color:#fff; }
  .go:hover { transform:translateY(-2px); box-shadow:0 10px 24px #0004; }
  .ghost { background:#eef0f6; color:#444; }
  .count { text-align:center; color:#666; font-size:13px; margin-top:10px; min-height:18px; }
  #result { margin-top:18px; }
  .hero { background:linear-gradient(135deg,#1f2330,#3a3f55); color:#fff; border-radius:18px;
          padding:22px 24px; text-align:center; animation:pop .45s cubic-bezier(.2,1.3,.4,1); }
  .hero .big { font-size:30px; font-weight:800; margin:6px 0; }
  .hero .pct { font-size:14px; opacity:.9; }
  .hero .why { font-size:13.5px; opacity:.85; margin-top:8px; }
  .runners { display:flex; gap:12px; margin-top:12px; flex-wrap:wrap; justify-content:center; }
  .mini { flex:1; min-width:170px; background:#fff; border-radius:14px; padding:14px 16px;
          box-shadow:0 6px 18px #0000001f; animation:pop .5s cubic-bezier(.2,1.3,.4,1); }
  .mini .nm { font-weight:700; font-size:16px; }
  .mini .gp { font-size:12px; color:#777; margin-top:2px; }
  .mini .pb { height:7px; border-radius:6px; background:#eee; margin-top:9px; overflow:hidden; }
  .mini .pf { height:100%; border-radius:6px; }
  @keyframes pop { from { opacity:0; transform:scale(.9) translateY(8px);} to {opacity:1; transform:none;} }
</style></head>
<body><div class="wrap">
  <h1>HEADER_TITLE</h1>
  <p class="sub">Tick the hobbies you already love - the model finds your next obsession.</p>
  <div class="panel">
    <div id="board"></div>
    <div class="bar">
      <button class="go" onclick="recommend()">Find my next hobby</button>
      <button class="ghost" onclick="surprise()">Surprise me</button>
      <button class="ghost" onclick="clearAll()">Clear</button>
    </div>
    <div class="count" id="count"></div>
    <div id="result"></div>
  </div>
</div>
<script>
const DATA = __DATA__;
const hobbies = Object.keys(DATA.vecs);
const DIM = DATA.vecs[hobbies[0]].length;
const selected = new Set();
const emojiOf = g => g.split(' ')[0];

function render(){
  const board = document.getElementById('board'); board.innerHTML = '';
  DATA.groups.forEach(g => {
    const head = document.createElement('div'); head.className = 'grp';
    head.innerHTML = '<span class="dot" style="background:'+DATA.group_color[g]+'"></span>'+g;
    const chips = document.createElement('div'); chips.className = 'chips';
    hobbies.filter(h => DATA.group_of[h] === g).forEach(h => {
      const c = document.createElement('div'); c.className = 'chip'; c.textContent = h;
      if (selected.has(h)) { c.classList.add('on'); c.style.background = DATA.group_color[g]; }
      c.onclick = () => { selected.has(h) ? selected.delete(h) : selected.add(h); render(); updateCount(); };
      chips.appendChild(c);
    });
    board.appendChild(head); board.appendChild(chips);
  });
}
function updateCount(){
  document.getElementById('count').textContent =
    selected.size ? selected.size + ' selected' : 'Pick at least one hobby to get a recommendation.';
}
function recommend(){
  const res = document.getElementById('result');
  if (selected.size === 0){ res.innerHTML = ''; updateCount(); return; }
  const q = new Array(DIM).fill(0);
  selected.forEach(h => { const v = DATA.vecs[h]; for (let i=0;i<DIM;i++) q[i]+=v[i]; });
  let n=0; for (let i=0;i<DIM;i++) n+=q[i]*q[i]; n=Math.sqrt(n)||1;
  for (let i=0;i<DIM;i++) q[i]/=n;
  const scored = hobbies.filter(h => !selected.has(h)).map(h => {
    const v = DATA.vecs[h]; let d=0; for (let i=0;i<DIM;i++) d+=q[i]*v[i]; return { h, s:d };
  }).sort((a,b)=>b.s-a.s);
  if (!scored.length){ res.innerHTML = '<p>Pick fewer hobbies to leave something to suggest!</p>'; return; }
  const top = scored.slice(0,3);
  const win = top[0].h, wv = DATA.vecs[win];
  const why = [...selected].map(h => { const v=DATA.vecs[h]; let d=0; for(let i=0;i<DIM;i++) d+=wv[i]*v[i]; return {h,d}; })
                .sort((a,b)=>b.d-a.d).slice(0,2).map(x=>x.h);
  const pct = s => Math.max(1, Math.round(s*100));
  let html = '<div class="hero">your next obsession could be...'
           + '<div class="big">'+emojiOf(DATA.group_of[win])+' '+win+'</div>'
           + '<div class="pct">'+pct(top[0].s)+'% match - '+DATA.group_of[win]+'</div>'
           + '<div class="why">because you like <b>'+why.join('</b> & <b>')+'</b></div></div>';
  if (top.length>1){
    html += '<div class="runners">';
    top.slice(1).forEach(t => {
      const col = DATA.group_color[DATA.group_of[t.h]];
      html += '<div class="mini"><div class="nm">'+emojiOf(DATA.group_of[t.h])+' '+t.h+'</div>'
            + '<div class="gp">'+DATA.group_of[t.h]+'</div>'
            + '<div class="pb"><div class="pf" style="width:'+pct(t.s)+'%;background:'+col+'"></div></div>'
            + '<div class="gp" style="margin-top:6px">'+pct(t.s)+'% match</div></div>';
    });
    html += '</div>';
  }
  res.innerHTML = html;
}
function clearAll(){ selected.clear(); render(); updateCount(); document.getElementById('result').innerHTML=''; }
function surprise(){
  selected.clear();
  const pool = hobbies.slice(); for (let i=pool.length-1;i>0;i--){ const j=Math.floor(Math.random()*(i+1)); [pool[i],pool[j]]=[pool[j],pool[i]]; }
  pool.slice(0,3).forEach(h => selected.add(h));
  render(); updateCount(); recommend();
}
render(); updateCount();
</script></body></html>'''

In [2]:
import os, urllib.request
import numpy as np, pandas as pd

CSV = "hobbies_hf.csv"
if not os.path.exists(CSV):
    url = "https://huggingface.co/datasets/alperugurcan/Hobbies/resolve/main/hobbies.csv"
    req = urllib.request.Request(url, headers={"User-Agent": "ydl-2026-student/1.0"})
    open(CSV, "wb").write(urllib.request.urlopen(req, timeout=60).read())
    print("downloaded", CSV)

df = pd.read_csv(CSV).drop_duplicates(subset="hobby").reset_index(drop=True)   # dataset has dup rows
hobbies = df["hobby"].tolist()
print(len(hobbies), "distinct hobbies (vocabulary straight from the data)")
print("columns:", list(df.columns))
df.head(3)

103 distinct hobbies (vocabulary straight from the data)
columns: ['hobby', 'tags', 'difficulty_level', 'cost', 'time_requirement', 'physical_activity_level', 'sociability', 'location', 'age_group', 'equipment_needed']


,hobby,tags,difficulty_level,cost,time_requirement,physical_activity_level,sociability,location,age_group,equipment_needed
0,Reading,"indoor,relaxing,educational",Beginner,Low,Daily,Low,Individual,Indoor,All Ages,Low
1,Jogging,"outdoor,fitness,cardio",Beginner,Low,Daily,Medium,Individual,Outdoor,All Ages,Low
2,Photography,"creative,visual,artistic",Intermediate,Medium,Weekly,Low,Mixed,Both,All Ages,Medium


## 1. Turn each hobby into a vector

Two feature blocks, concatenated:
- **tags** -> tf-idf (so distinctive tags weigh more than common ones), splitting on commas;
- **categorical attributes** -> one-hot, down-weighted a touch so the free-text tags lead.

Each row is L2-normalised, so a plain dot product equals cosine similarity.

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from scipy.sparse import hstack

CATS = ["difficulty_level", "cost", "time_requirement", "physical_activity_level",
        "sociability", "location", "age_group", "equipment_needed"]

tags = TfidfVectorizer(token_pattern=r"[^,]+").fit_transform(df["tags"].fillna(""))
oh   = OneHotEncoder().fit_transform(df[CATS].fillna("NA"))
X    = hstack([tags, oh * 0.5]).toarray()
X    = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)   # unit rows -> dot == cosine
vec  = {h: X[i] for i, h in enumerate(hobbies)}
print("feature vector per hobby:", X.shape[1], "dimensions")

feature vector per hobby: 106 dimensions


## 2. Do the neighbours make sense?

A hobby's nearest neighbours by cosine should feel alike in spirit.

In [4]:
def neighbors(h, k=5):
    s = X @ vec[h]
    return [hobbies[j] for j in np.argsort(-s) if hobbies[j] != h][:k]

for h in ["Photography", "Jogging", "Knitting", "Chess", "Woodworking"]:
    if h in vec:
        print(f"{h:14s} -> {neighbors(h)}")

Photography    -> ['Painting', 'Digital Art', 'Watercolor Painting', 'Sketching', 'Drawing']
Jogging        -> ['Running', 'Cycling', 'Yoga', 'Hiking', 'Ultimate Frisbee']
Knitting       -> ['Origami', 'Scrapbooking', 'Drawing', 'Sketching', 'Watercolor Painting']
Chess          -> ['Board Games', 'Scrabble', 'Acroyoga', 'Ukulele', 'Virtual Reality Gaming']
Woodworking    -> ['Carpentry', 'Pottery Making', 'Pottery', 'Sculpting', 'Leatherworking']


## 3. The recommender

Average the vectors of the hobbies you like, then return the nearest hobbies you don't do yet.

In [5]:
def recommend(liked, topn=5):
    liked = [h for h in liked if h in vec]
    if not liked:
        return []
    q = np.mean([vec[h] for h in liked], axis=0)
    q = q / (np.linalg.norm(q) + 1e-9)
    s = X @ q
    return [(hobbies[j], round(float(s[j]), 3)) for j in np.argsort(-s) if hobbies[j] not in liked][:topn]

for likes in [["Photography", "Painting"], ["Jogging", "Yoga"], ["Knitting", "Cooking"],
              ["Chess", "Reading"]]:
    print(f"{str(likes):32s} -> {recommend(likes)}")

['Photography', 'Painting']      -> [('Digital Art', 0.78), ('Sketching', 0.696), ('Watercolor Painting', 0.696), ('Drawing', 0.696), ('Pottery Making', 0.653)]
['Jogging', 'Yoga']              -> [('Running', 0.816), ('Cycling', 0.632), ('Juggling', 0.597), ('Reading', 0.597), ('Mindfulness Meditation', 0.574)]
['Knitting', 'Cooking']          -> [('Origami', 0.828), ('Baking', 0.72), ('Reading', 0.675), ('Scrapbooking', 0.663), ('Drawing', 0.654)]
['Chess', 'Reading']             -> [('Board Games', 0.688), ('Knitting', 0.685), ('Origami', 0.685), ('Scrabble', 0.675), ('Coin Collecting', 0.624)]


## 4. Group the hobbies + an interactive map

KMeans finds families in the feature space (no hand labels); PCA drops the vectors to 2D for an
interactive Plotly map - each point a hobby, colour its discovered family.

In [6]:
import pandas as pd
import plotly.express as px
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

K = 6
km = KMeans(n_clusters=K, n_init=10, random_state=0).fit(X)

def central(c):
    members = [i for i in range(len(hobbies)) if km.labels_[i] == c]
    members.sort(key=lambda i: float(X[i] @ km.cluster_centers_[c]), reverse=True)
    return [hobbies[i] for i in members[:2]]

# human-readable family names, anchored to a hobby unique to each cluster
# (anchor-based so the labels follow their cluster even if the data changes)
ANCHORS = [
    ("Martial Arts",   "🥊 Sports & competition"),
    ("Leatherworking", "🎨 Arts & crafts"),
    ("Hiking",         "🏃 Active & movement"),
    ("Glassblowing",   "🔧 Maker & build"),
    ("Origami",        "🧘 Calm & creative"),
    ("Scuba Diving",   "🌊 Adventure & thrills"),
]

def family_name(c):
    members = {hobbies[i] for i in range(len(hobbies)) if km.labels_[i] == c}
    for anchor, label in ANCHORS:
        if anchor in members:
            return label
    return "✨ " + " · ".join(central(c))          # fallback for any unanchored cluster

label_of = {c: family_name(c) for c in range(K)}
group    = [label_of[c] for c in km.labels_]

print("discovered hobby families (human-named):")
for c in range(K):
    print(f"  {label_of[c]:24s}: {[hobbies[i] for i in range(len(hobbies)) if km.labels_[i]==c][:8]} ...")

coords = PCA(n_components=2, random_state=0).fit_transform(X)
dfmap = pd.DataFrame({"x": coords[:, 0], "y": coords[:, 1], "hobby": hobbies, "family": group})
fig = px.scatter(dfmap, x="x", y="y", color="family", hover_name="hobby",
                 title="Hobby map - content-based (colour = family the model discovered)",
                 hover_data={"x": False, "y": False, "family": True})
fig.update_traces(marker=dict(size=10, line=dict(width=0.4, color="white")))
fig.update_layout(width=950, height=680, legend_title_text="family", plot_bgcolor="#fafafa")
fig.update_xaxes(visible=False); fig.update_yaxes(visible=False)
fig.write_html("outputs/hobby_map_content.html")
print("saved -> outputs/hobby_map_content.html")
fig.show()

discovered hobby families (human-named):
  🥊 Sports & competition  : ['Archery', 'Weightlifting', 'Martial Arts', 'Stand-up Comedy', 'Fitness Gaming', 'CrossFit', 'Cycling', 'Tennis'] ...
  🎨 Arts & crafts         : ['Photography', 'Cooking', 'Model Building', 'Pottery', 'Painting', 'Pottery Making', 'Amateur Astronomy', 'Leatherworking'] ...
  🏃 Active & movement     : ['Jogging', 'Gardening', 'Hiking', 'Swimming', 'Running', 'Dancing', 'Geocaching', 'Acroyoga'] ...
  🔧 Maker & build         : ['Woodworking', 'Sculpting', 'Carpentry', 'Astrophotography', '3D Printing', 'Model Railroading', 'Glassblowing', 'Virtual Reality Gaming'] ...
  🧘 Calm & creative       : ['Reading', 'Yoga', 'Knitting', 'Board Games', 'Calligraphy', 'Origami', 'Singing', 'Scrapbooking'] ...
  🌊 Adventure & thrills   : ['Mountain Biking', 'Kayaking', 'Rock Climbing', 'Surfing', 'Drone Flying', 'Beekeeping', 'Golf', 'Scuba Diving'] ...
saved -> outputs/hobby_map_content.html


## 5. The fun artifact: the Hobby Matchmaker game

Bake the vectors + families into a standalone HTML page: tick hobbies you like, get a
recommendation with a match score and a "because you like..." explanation. Runs fully offline.

In [7]:
import json, os
os.makedirs("outputs", exist_ok=True)

groups_order = list(dict.fromkeys(group))
palette      = ["#6C8EF5", "#E8743B", "#1DBC9B", "#E0529C", "#9B5DE5", "#F2B705", "#3CA0A0"]
group_color  = {g: palette[i % len(palette)] for i, g in enumerate(groups_order)}
vecs_json    = {h: [round(float(x), 4) for x in vec[h]] for h in hobbies}

DATA = {"vecs": vecs_json, "group_of": {hobbies[i]: group[i] for i in range(len(hobbies))},
        "groups": groups_order, "group_color": group_color}

html = GAME_TEMPLATE.replace("HEADER_TITLE", "Hobby Matchmaker").replace("__DATA__", json.dumps(DATA))
with open("outputs/hobby_matchmaker_content.html", "w") as f:
    f.write(html)
print("saved -> outputs/hobby_matchmaker_content.html  (", len(html), "bytes,", len(hobbies), "hobbies )")

saved -> outputs/hobby_matchmaker_content.html  ( 72067 bytes, 103 hobbies )


## What I learned

- **A ready dataset removes the hand-curation problem.** The 117 hobbies and their descriptions
  come straight from Hugging Face, so the vocabulary is data-driven - the thing the Reddit
  notebook couldn't avoid.
- **Content-based vs behavioural similarity are different lenses.** Here "similar" means *alike
  in character* (creative / cheap / indoor), built with **tf-idf + one-hot + cosine** - no
  Word2Vec needed, because there are no people to co-occur. The Reddit notebook's "similar"
  means *the same people do them*.
- **The toolbox transfers.** tf-idf and cosine similarity - straight from today's lab - are
  enough to power both a recommender and a playable game.